# Stage 4.5 v4 — TrackNet ball detector (Colab GPU training)

Trains the temporal ball detector on the prepared 720p frame caches.
**Runtime → Change runtime type → GPU (T4 or A100).**

## Upload layout (to Google Drive, e.g. `MyDrive/pb_v4/`)
```
pb_v4/
  repo/            <- the repo (needs stages/track_ball + stages/finetune_ball_model)
  data/
    pb_2min/  v4_manifest.json + frames_720/   (from prepare_v4.py)
    pb_3min/  ...
    pb_4min/  ...
    pb_5min/  ...
```
Only the small `v4_manifest.json` + `frames_720/` JPEGs are needed — NOT the 4K videos.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/pb_v4')   # <-- adjust if needed
REPO = ROOT / 'repo'
DATA = ROOT / 'data'
sys.path.insert(0, str(REPO))
print('repo exists:', REPO.exists(), '| data exists:', DATA.exists())

In [ ]:
# OpenCV + torch are preinstalled on Colab; install only if missing
try:
    import cv2  # noqa
except Exception:
    !pip -q install opencv-python-headless

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from stages.finetune_ball_model.train_v4 import TrainConfig
from stages.finetune_ball_model._v4_cache import train_from_manifests

# DECISION: hold out the most visually-distinct clip as the generalization test.
HOLDOUT = 'pb_5min'   # <-- set to the clip you flagged as most distinct
ALL = ['pb_2min','pb_3min','pb_4min','pb_5min']
train_folders = [DATA/c for c in ALL if c != HOLDOUT]
holdout_folder = DATA/HOLDOUT
print('train:', [f.name for f in train_folders], '| holdout:', holdout_folder.name)

In [ ]:
cfg = TrainConfig(
    epochs=30,
    batch_size=8,        # lower to 4 on T4 if OOM; raise on A100
    lr=1e-3,
    out_path=str(ROOT/'ball_model_v4.pt'),   # saved to Drive (persists)
)
result = train_from_manifests(train_folders, holdout_folder, cfg, num_workers=4)
print('BEST held-out recall:', result['best_held_recall'])

## Result
- `ball_model_v4.pt` + `validation_report.json` are saved to `MyDrive/pb_v4/`.
- **Acceptance bar: held-out recall ≥ 0.80.** If below, first raise input
  resolution to 1080p (set `PROC_H,PROC_W=1088,1920` in `_v4_data.py`) and
  re-prepare + retrain, then escalate the architecture if still short.
- Download `ball_model_v4.pt` back to `data/models/` locally; the rest
  (Stage 4 inference + trajectory + re-running 5–11) runs on the local machine.

In [ ]:
import json
rep = json.load(open(ROOT/'validation_report.json'))
print('holdout clip:', rep['holdout_clip'], '| best recall:', rep['best_held_recall'])
for h in rep['history'][-5:]:
    print(h)